# Notebook 22 — Reasoning Post-Training with GRPO and Verifiable Rewards

    ## Learning objectives

    - Derive group-relative advantages and distinguish GRPO from SFT, DPO, and PPO
- Design auditable outcome, format, and process rewards without rewarding shortcuts
- Configure a guarded Hugging Face TRL reasoning run and evaluate quality against inference cost

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'datasets>=3.5,<6', 'peft>=0.15', 'trl>=0.16', 'accelerate>=1.6', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if True and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 22.1 From imitation to exploration

SFT increases likelihood of demonstrations; DPO learns from fixed chosen/rejected pairs. Online RL
samples completions from the current policy, scores them, and changes the policy toward high-reward
behavior. This can discover solutions outside a static dataset, but it also exposes the optimizer to
every flaw in the reward. Use RL when exploration matters and rewards can be independently audited—not
because a task is fashionable or difficult.

Group Relative Policy Optimization (GRPO) samples a group of completions for each prompt and centers or
standardizes reward within that group. Relative advantages remove the need for a separately trained
value model used by PPO-like methods. The reference/KL mechanism or clipping limits destructive policy
drift. Group estimates are noisy when completions are correlated or all rewards are identical.


In [ ]:
import torch
rewards = torch.tensor([[1.0, 0.0, 0.5, 1.0], [0.0, 0.0, 0.0, 0.0]])
mean = rewards.mean(-1, keepdim=True)
std = rewards.std(-1, keepdim=True, unbiased=False).clamp_min(1e-4)
advantages = (rewards - mean) / std
print("rewards:\n", rewards, "\nadvantages:\n", advantages)
print("zero-variance group carries no ranking information")


## 22.2 Verifiers and reward contracts

Math answers, compiled code, unit tests, games, and formal constraints permit outcome rewards. Parse a
clearly delimited final answer, normalize only equivalences you truly accept, and keep the checker
isolated from untrusted code. Partial-credit rules must be monotonic and difficult to exploit. Format
rewards should be small relative to correctness so the model cannot win by producing pristine wrappers
around wrong content.

Reward hacking occurs when the proxy is easier than the intended task: leaking tests, matching a magic
string, exploiting numerical tolerances, returning no-op code, or producing long judge-pleasing text.
Maintain hidden adversarial tests, mutate problem representations, compare independent verifiers, and
log raw completions with reward components. Treat verifier changes as dataset and objective changes.


In [ ]:
import re
def exact_math_reward(completions, answer, **_):
    scores = []
    for completion, expected in zip(completions, answer):
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        found = re.findall(r"FINAL:\s*(-?\d+(?:\.\d+)?)", text)
        scores.append(float(bool(found) and float(found[-1]) == float(expected)))
    return scores
probes = [[{"role": "assistant", "content": "work... FINAL: 391"}],
          [{"role": "assistant", "content": "FINAL: 390"}]]
print(exact_math_reward(probes, ["391", "391"]))


## 22.3 Policy objective and stability

For sampled token actions, a policy-gradient surrogate multiplies log-probability ratios by advantages.
Clipping prevents one batch from making arbitrarily large changes. A KL penalty against a reference
policy preserves capabilities but can also limit exploration. Token-level normalization choices affect
whether long completions dominate. Reward scale, group size, number of generations, temperature,
maximum completion length, and effective prompt batch are coupled hyperparameters.

Monitor reward components and distributions, reward standard deviation, fraction of zero-variance groups,
KL, clip ratio, entropy, completion length, invalid-format rate, pass@1/pass@k, tokens per accepted solution,
and held-out capability/safety. A rising training reward with flat hidden-test accuracy is a verifier leak
or overfitting alarm.


In [ ]:
old_logp = torch.tensor([-1.2, -0.8, -2.0])
new_logp = torch.tensor([-1.0, -1.1, -1.7])
adv = torch.tensor([1.0, -0.5, 0.8])
ratio = (new_logp - old_logp).exp(); epsilon = 0.2
surrogate = torch.minimum(ratio * adv, ratio.clamp(1-epsilon, 1+epsilon) * adv)
print({"ratio": ratio.tolist(), "clipped_objective": surrogate.mean().item()})


## 22.4 A guarded TRL configuration

TRL's `GRPOTrainer` accepts standard or conversational prompts and one or more reward functions. Extra
dataset columns are forwarded to custom rewards. Current TRL also supports tools and stateful environment
factories, but those interfaces evolve; pin the tested version and consult its documentation. The example
below is deliberately opt-in because online generation and multiple completions are materially more
expensive than a small SFT step.


In [ ]:
RUN_GRPO = False
if RUN_GRPO:
    from datasets import Dataset
    from trl import GRPOConfig, GRPOTrainer
    data = Dataset.from_list([
        {"prompt": "Compute 17*23. End with FINAL: number", "answer": "391"},
        {"prompt": "Compute 29*14. End with FINAL: number", "answer": "406"},
    ] * 32)
    config = GRPOConfig(output_dir="artifacts/qwen-grpo", max_steps=10,
                        num_generations=4, max_completion_length=128,
                        learning_rate=5e-6, report_to="none", log_completions=True)
    trainer = GRPOTrainer(model="Qwen/Qwen2.5-0.5B-Instruct", reward_funcs=exact_math_reward,
                          args=config, train_dataset=data)
    trainer.train()
else:
    print("GRPO skipped. Audit rewards, choose a GPU runtime, then opt in.")


## 22.5 Reasoning evaluation and inference-time compute

Do not grade exposed reasoning prose as if it were faithful cognition. Score final outcomes, robustness
to irrelevant details and representation changes, calibration, safety, and resource use. Report pass@1
and pass@k with the sampling configuration, plus the selector or verifier used to choose an answer.
Best-of-N improves only when samples contain useful diversity and the selector identifies correctness.

Compare the RL checkpoint against its SFT parent on frozen target and retention suites. Analyze problems
by difficulty and verifier type. Measure generated reasoning tokens, latency, and energy per solved task.
Red-team answer extraction, delimiter spoofing, test leakage, grader timeouts, and sandbox escapes. Store
reward-code revision, environment image, model/reference revisions, rollout configuration, and raw
evaluation traces in the run record.


## 22.6 Group-relative advantage mechanics

GRPO samples a group of completions for each prompt and normalizes rewards within that group. Relative advantages avoid a separate value model but become unstable when every reward is equal or a verifier is sparse. Log within-group reward variance, valid-completion rate, KL to the reference, response length, and per-verifier outcomes. The group is a Monte Carlo sample whose diversity depends on decoding; duplicated completions waste compute and distort normalization. Handle zero variance explicitly.


In [ ]:
rewards=torch.tensor([1.,0.,1.,0.]); mean=rewards.mean(); std=rewards.std(unbiased=False); advantages=(rewards-mean)/std.clamp_min(1e-6); print(mean.item(),std.item(),advantages,advantages.mean())


## 22.7 Reward-hacking test design

A verifiable reward is only as strong as its parser, sandbox, tests, and hidden cases. Attack formatting, numerical tolerances, timeouts, nondeterminism, resource use, test leakage, and alternate encodings. Separate invalid outputs from incorrect valid answers and retain adversarial completions. Evaluate pass@k, unique solutions, robustness under new hidden tests, and general retention. Keep authorization and safety constraints outside model-generated reasoning or rewards.


In [ ]:
def strict_integer_reward(text,expected):
 import re
 if not re.fullmatch(r"-?(0|[1-9][0-9]*)",text.strip()): return 0.0
 return float(int(text)==expected)
for answer in ["42","42.0","Answer: 42","0042","41"]: print(repr(answer),strict_integer_reward(answer,42))


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [DeepSeekMath / GRPO](https://arxiv.org/abs/2402.03300)
- [TRL GRPOTrainer](https://huggingface.co/docs/trl/grpo_trainer)


## Exercises

    1. Design three attacks against the exact-answer reward and harden its parser.
2. Compare group sizes using reward variance, pass@1, and generated tokens per update.
3. Create a frozen hidden-test gate that would detect reward hacking before promotion.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
